# Fitting a transit on an active young star

This notebook needs **no data files and no network** — it builds a light curve
of a 20 Myr star and fits it, so the failure modes PetCare is built around are
visible directly.

The star rotates in 1.4 days with a few percent of spot modulation, flares, and
has a real 4200 ppm transit. Two TESS sectors a year apart, with different spot
amplitudes in each — which is the normal situation, not a contrived one.

What it shows:

1. why flattening the light curve first destroys the transit,
2. the simultaneous variability + transit fit that replaces it,
3. the case where a global Gaussian process explains the transit away, and the
   local-baseline fit that recovers it,
4. one variability model per sector.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from petcare import ephem
from petcare.fitting import (bin_for_fit, fit_gp_hyperparameters,
                             fit_gp_hyperparameters_segmented, gp_trend_masked,
                             joint_fit, local_joint_fit, transit_model)
from petcare.stellar import a_over_rs, duration_t14

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25})
rng = np.random.default_rng(20260921)

## 1. Build the star

Two sectors of 2-minute cadence, 25 days each, separated by a year. The spot
amplitude drops from 2.2% to 0.9% between them and the phase moves: a 20 Myr
star's surface does not look the same a year later.

In [2]:
P_ROT = 1.4                 # rotation period [d]
PERIOD, T0 = 6.9, 3.0       # transit ephemeris [d], [d]
RP_RS, A_RS, B = 0.065, 13.0, 0.30
WHITE = 1200e-6             # point-to-point scatter at 2 minutes

# spot modulation: the rotation period plus its first harmonic
def spots(t, amp, phase):
    return amp * (np.sin(2 * np.pi * (t / P_ROT + phase))
                  + 0.45 * np.sin(4 * np.pi * (t / P_ROT + phase) + 1.1))

# exponential-decay flares, at a young star's rate
def flares(t, rate_per_day=1.6):
    out = np.zeros(t.size)
    n = rng.poisson(rate_per_day * np.ptp(t))
    for _ in range(n):
        t_peak = rng.uniform(t.min(), t.max())
        amp = 10 ** rng.uniform(-3.4, -1.8)
        tau = 10 ** rng.uniform(-3.2, -2.2)
        rise = (t > t_peak - tau / 3) & (t <= t_peak)
        decay = t > t_peak
        out[rise] += amp * (t[rise] - (t_peak - tau / 3)) / (tau / 3)
        out[decay] += amp * np.exp(-(t[decay] - t_peak) / tau)
    return out

cad = 2.0 / 60 / 24
t = np.concatenate([np.arange(0.0, 25.0, cad), np.arange(380.0, 405.0, cad)])
sector = np.where(t < 200, 7, 34)          # per-point sector label

variability = np.where(t < 200, spots(t, 0.022, 0.0), spots(t, 0.009, 0.37))
transit = transit_model(t, T0, PERIOD, RP_RS, A_RS, B, exp_time_days=cad)
f = transit + variability + flares(t) + rng.normal(0, WHITE, t.size)
err = np.full(t.size, WHITE)

DUR = duration_t14(PERIOD, A_RS, RP_RS, B)
DEPTH_TRUE = (1 - transit.min()) * 1e6
n_transits = int(np.ptp(ephem.epoch_number(t, PERIOD, T0)[
    np.abs(ephem.phase(t, PERIOD, T0)) < 0.5 * DUR]) + 1)
print(f"true depth {DEPTH_TRUE:.0f} ppm, T14 {DUR * 24:.2f} h, "
      f"spot amplitude 22000 -> 9000 ppm, {t.size} cadences")

true depth 5027 ppm, T14 4.15 h, spot amplitude 22000 -> 9000 ppm, 36000 cadences


The transit is a factor of five below the variability. This is the ordinary
case for this sample, not the hard end of it.

In [3]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5))
for ax, lo, hi, name in ((axes[0], 0, 25, "sector 7"), (axes[1], 380, 405, "sector 34")):
    m = (t >= lo) & (t <= hi)
    ax.plot(t[m], f[m], ".", ms=0.7, color="0.45", rasterized=True)
    for k in range(-2, 60):
        tc = T0 + k * PERIOD
        if lo < tc < hi:
            ax.axvline(tc, color="tab:red", lw=0.8, alpha=0.6)
    ax.set_ylabel("flux")
    ax.set_title(f"{name} (transits marked)", loc="left")
axes[1].set_xlabel("time [d]")
fig.tight_layout()

## 2. What flattening does

The standard move is to remove the variability with a running median and fit
the transit afterwards. There is no window that works: wide enough to leave the
transit alone leaves the variability too, and narrow enough to remove a 1.4-day
modulation removes a 3-hour transit along with it.

In [4]:
def running_median(t, f, window_days):
    out = np.empty(f.size)
    idx = np.argsort(t)
    ts, fs = t[idx], f[idx]
    lo = np.searchsorted(ts, ts - window_days / 2)
    hi = np.searchsorted(ts, ts + window_days / 2)
    for i in range(ts.size):
        out[i] = np.median(fs[lo[i]:hi[i]])
    res = np.empty(f.size); res[idx] = out
    return res

ph = ephem.phase(t, PERIOD, T0)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, w in zip(axes, (0.2, 0.5, 2.0)):
    flat = f - running_median(t, f, w) + 1.0
    sel = np.abs(ph) < 3 * DUR
    xb = np.linspace(-3 * DUR, 3 * DUR, 45)
    ib = np.digitize(ph[sel], xb)
    yb = np.array([np.median(flat[sel][ib == k]) if (ib == k).sum() > 5 else np.nan
                   for k in range(1, xb.size)])
    depth = (1 - np.nanmin(yb)) * 1e6
    ax.plot(ph[sel] * 24, (flat[sel] - 1) * 1e6, ".", ms=0.6, color="0.7", rasterized=True)
    ax.plot(0.5 * (xb[1:] + xb[:-1]) * 24, (yb - 1) * 1e6, "o-", ms=3, color="tab:blue")
    ax.axhline(-DEPTH_TRUE, color="tab:red", ls="--", lw=1)
    ax.set_title(f"{w * 24:.0f} h window\nrecovered {depth:.0f} of {DEPTH_TRUE:.0f} ppm")
    ax.set_xlabel("hours from mid-transit")
axes[0].set_ylabel("flux - 1 [ppm]")
axes[0].set_ylim(-9000, 6000)
fig.tight_layout()

The dashed red line is the true depth. The short windows eat the transit; the
long one leaves so much variability that the fold is not usable.

## 3. Fit the variability and the transit at the same time

Two steps, and the order matters:

1. **Fit the variability hyperparameters with the transit masked**, so the
   kernel describes the star and not the candidate. `fit_gp_hyperparameters`
   also detects and masks flares internally — without that the kernel is driven
   to maximum flexibility trying to explain them, which is exactly the shape
   that then swallows a transit.
2. **Hold those hyperparameters fixed** and fit the transit jointly. The
   hyperparameter likelihood is nearly flat along an (amplitude, quality
   factor) degeneracy on stars like this, so a fit free to move along it buys
   flexibility for free and pays for it with the transit depth.

In [5]:
tb, fb, eb, nb_, info = bin_for_fit(t, f, err, 600.0)      # 10-minute bins
mask = ephem.in_transit(tb, PERIOD, T0, DUR, width=2.5)

gp_spec = fit_gp_hyperparameters(tb, fb, mask_out=mask,
                                 min_rho_days=3 * DUR, prot=P_ROT)
print(gp_spec.kind, "| amplitude %.0f ppm" % (gp_spec.amplitude() * 1e6),
      "| shortest timescale %.2f d = %.1f durations"
      % (gp_spec.timescale(), gp_spec.timescale() / DUR))
print("flare masking:", gp_spec.detail.get("flare_masking"))

rotation | amplitude 13720 ppm | shortest timescale 0.72 d = 4.1 durations
flare masking: {'applied': True, 'n_events': 6, 'masked_fraction': 0.006111111111111111, 'amplitude_before': 0.01369065624735841, 'amplitude_after': 0.013719981173309942}


In [6]:
rho_star = 1.6                                   # g/cm3, a young K dwarf
trend = gp_trend_masked(tb, fb, eb, mask, gp_spec)
depth_guess = float(np.median((fb - trend + 1)[~mask & (np.abs(
    ephem.phase(tb, PERIOD, T0)) > 0.75 * DUR)])
    - np.median((fb - trend + 1)[np.abs(ephem.phase(tb, PERIOD, T0)) < 0.4 * DUR]))

fit = joint_fit(tb, fb, eb, PERIOD, T0, DUR, rho_star=rho_star,
                gp_prior=gp_spec, exp_time_days=600 / 86400,
                depth_guess=depth_guess)
print(f"depth  {fit.depth_ppm:8.0f} ppm   (true {DEPTH_TRUE:.0f})")
print(f"T14    {fit.t14_h:8.2f} h     (true {DUR * 24:.2f})")
print(f"Rp/R*  {fit.rp_rs:8.4f}       (true {RP_RS})")
print(f"b      {fit.b:8.2f}       (true {B})")
print(f"delta lnL vs no transit: {fit.delta_logl:.0f}")

depth      5204 ppm   (true 5027)
T14        4.20 h     (true 4.15)
Rp/R*    0.0665       (true 0.065)
b          0.38       (true 0.3)
delta lnL vs no transit: 514


The variability model has to be seen to be believed: if it is tracking the
transits rather than the star, the depth is wrong no matter how good the
formal errors look. PetCare plots this for every fit, and so should you.

In [7]:
from petcare.fitting import gp_predict

model, gp_mean = gp_predict(tb, fb, eb, fit)
m = tb < 25
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(tb[m], fb[m], ".", ms=1.2, color="0.6", rasterized=True)
axes[0].plot(tb[m], gp_mean[m] + 1, "-", lw=1, color="tab:orange",
             label="variability model")
axes[0].legend(loc="upper right"); axes[0].set_ylabel("flux")
axes[1].plot(tb[m], (fb[m] - gp_mean[m] - 1) * 1e6, ".", ms=1.2, color="0.6",
             rasterized=True)
axes[1].plot(tb[m], (model[m] - 1) * 1e6, "-", lw=1.2, color="tab:red",
             label="transit model")
axes[1].legend(loc="lower right")
axes[1].set_ylabel("flux - 1 [ppm]"); axes[1].set_xlabel("time [d]")
fig.tight_layout()

In [8]:
phb = ephem.phase(tb, fit.period, fit.t0)
sel = np.abs(phb) < 3 * DUR
resid = fb - gp_mean
xb = np.linspace(-3 * DUR, 3 * DUR, 60)
ib = np.digitize(phb[sel], xb)
yb = np.array([np.mean(resid[sel][ib == k]) if (ib == k).sum() > 3 else np.nan
               for k in range(1, xb.size)])
fine = np.linspace(-3 * DUR, 3 * DUR, 600)

plt.figure(figsize=(6, 3.6))
plt.plot(phb[sel] * 24, (resid[sel] - 1) * 1e6, ".", ms=1, color="0.75", rasterized=True)
plt.plot(0.5 * (xb[1:] + xb[:-1]) * 24, (yb - 1) * 1e6, "o", ms=4, color="tab:blue")
plt.plot(fine * 24, (transit_model(fine + fit.t0, fit.t0, fit.period, fit.rp_rs,
                                   fit.a_rs, fit.b) - 1) * 1e6, "-", lw=1.4,
         color="tab:red")
plt.axhline(-DEPTH_TRUE, color="k", ls=":", lw=1)
plt.xlabel("hours from mid-transit"); plt.ylabel("flux - 1 [ppm]")
plt.title("phase fold, variability removed")
plt.tight_layout()

## 4. The other variability model, and why there are two

On the most active stars the fit above fails in a way that looks like success:
the marginalised likelihood genuinely prefers *no transit*, because a flexible
enough kernel can draw the dips itself. On AU Mic in the real sample the
preference is 617 in log likelihood, against a transit that is obvious by eye.

The alternative is a **local polynomial baseline per transit window**, fitted
simultaneously with the transit — the "notch" form. A quadratic over a couple
of transit durations cannot follow a three-hour dip, so it cannot absorb the
signal.

It has its own failure, and this star triggers it: the window must be **narrow
compared with the rotation period**. Here Prot is 1.4 days and the transit is
4.2 hours, so the default half-window of three durations spans Prot/2.7 — the
baseline follows the star and the transit is whatever is left over. Below,
first without the guard and then with it.

In [9]:
from petcare.fitting import local_window_durations

wide = local_joint_fit(tb, fb, eb, PERIOD, T0, DUR, rho_star=rho_star,
                       exp_time_days=600 / 86400, window_durations=3.0,
                       poly_order=2, depth_guess=depth_guess)
print(f"3.0 durations = Prot/{P_ROT / (3.0 * DUR):.1f}: "
      f"depth {wide.depth_ppm:.0f} ppm, T14 {wide.t14_h:.2f} h, "
      f"b {wide.b:.2f}, at a bound: {wide.detail['railed']}")

3.0 durations = Prot/2.7: depth 3 ppm, T14 0.23 h, b 0.98, at a bound: ['t0', 'period', 'rp_rs', 'a_rs', 'b']


In [10]:
# passing the rotation period caps the window automatically
fit_local = local_joint_fit(tb, fb, eb, PERIOD, T0, DUR, rho_star=rho_star,
                            exp_time_days=600 / 86400, window_durations=3.0,
                            poly_order=2, depth_guess=depth_guess, prot=P_ROT)
print(fit_local.detail["window_note"], "\n")

print(f"{'':10s}{'depth [ppm]':>13s}{'T14 [h]':>10s}{'b':>7s}")
print(f"{'truth':10s}{DEPTH_TRUE:13.0f}{DUR * 24:10.2f}{B:7.2f}")
print(f"{'GP':10s}{fit.depth_ppm:13.0f}{fit.t14_h:10.2f}{fit.b:7.2f}")
print(f"{'local':10s}{fit_local.depth_ppm:13.0f}{fit_local.t14_h:10.2f}{fit_local.b:7.2f}")

local baseline window narrowed from 3 to 1.62 transit durations: Prot = 1.400 d, and a polynomial over more than Prot/5 follows the star instead of levelling the window 

            depth [ppm]   T14 [h]      b
truth              5027      4.15   0.30
GP                 5204      4.20   0.38
local              4213      4.13   0.43


`pc_fit` runs both and arbitrates by measurement, not preference. A fit is
disqualified outright if it is not a measurement — no convergence, a
non-positive depth, no preference for a transit over no transit, or a shape
parameter sitting on a bound, which is what happened to the wide-window fit
above. Of the survivors, the GP is kept unless it falls a factor of two short
of the local fit's depth: a polynomial baseline cannot absorb a transit, so if
they disagree by that much, the GP has.

In [11]:
from petcare.scripts.pc_fit import _choose_fit

for label, local_arm in (("wide window", wide), ("capped window", fit_local)):
    chosen, why = _choose_fit("auto", fit, local_arm, depth_guess)
    print(f"{label:15s} -> {why['chosen']}: {why['reason'][:150]}")

wide window     -> gp: the local fit is unusable (the model with a transit is no better than the one without (delta lnL = 0.0))
capped window   -> gp: the two models agree on the depth (5204 vs 4213 ppm); kept the GP


## 5. One variability amplitude per sector — an option, and a warning

A single amplitude across sectors a year apart describes neither of them. Here
the spot amplitude falls by a factor of 2.4 between the two sectors, so the
shared model is too flexible for sector 34.

`fit_gp_hyperparameters_segmented` fits one kernel per sector and keeps a
single transit shared across all of them, so the transit is still constrained
by every sector at once. Only the **amplitude** is free per segment: the
rotation period and the quality factor stay pinned to the global fit, because
those are the parameters that decide whether a kernel *can* follow a
three-hour dip, and a quarter of the data is not enough to pin them.

In [12]:
labels = np.where(tb < 200, 7, 34)
seg = fit_gp_hyperparameters_segmented(
    tb, fb, mask_out=mask, labels=labels, min_rho_days=3 * DUR, prot=P_ROT,
    fallback=gp_spec)

print(seg.kind, "|", seg.reason)
print("free per segment:", seg.detail["free_per_segment"],
      "| pinned:", seg.specs[0].detail.get("pinned"))
for ids, amp in zip(seg.ids, seg.detail["amplitude_per_segment"]):
    print(f"  sector {ids}: amplitude {amp * 1e6:8.0f} ppm")
print(f"  one global model: {gp_spec.amplitude() * 1e6:.0f} ppm")

fit_seg = joint_fit(tb, fb, eb, PERIOD, T0, DUR, rho_star=rho_star,
                    gp_prior=seg, exp_time_days=600 / 86400,
                    depth_guess=depth_guess)
print(f"\n{'':12s}{'depth [ppm]':>13s}{'T14 [h]':>10s}")
print(f"{'truth':12s}{DEPTH_TRUE:13.0f}{DUR * 24:10.2f}")
print(f"{'one GP':12s}{fit.depth_ppm:13.0f}{fit.t14_h:10.2f}")
print(f"{'per sector':12s}{fit_seg.depth_ppm:13.0f}{fit_seg.t14_h:10.2f}")

rotation per segment | 2 of 2 segments fitted on their own data
free per segment: amplitude | pinned: ['log_period', 'log_Q0']
  sector [7]: amplitude    25054 ppm
  sector [34]: amplitude    10626 ppm
  one global model: 13720 ppm



              depth [ppm]   T14 [h]
truth                5027      4.15
one GP               5204      4.20
per sector              1      0.07


**On this star, the split costs the transit.** Freeing the amplitude lets
sector 7's kernel go to 25000 ppm, above the star's own 17000 ppm of spot
modulation, and a kernel with more variance than the data has will happily
explain a 5000 ppm dip as well.

On the nine real validation planets it is neither a disaster nor a gain — the
summed fractional depth error against the literature is 128% with one global
model and 123% per sector, which on nine planets is a wash. So the option
exists, is measured, and is **off by default**:

```bash
pc-fit ... --gp-segments sector --segment-free amplitude
```

One thing it definitely breaks, found the hard way: never run the **period
search** on a segment-detrended series. A per-segment model removes slightly
different things from each sector, the box SNR of a transit split across them
drops, and the search walks off — which loses the planet in both fit arms at
once. `pc_fit` refines the ephemeris on the global model and hands the
segmented one only to the transit fit.

A useful general rule from that: when both variability models fail at once,
suspect the ephemeris, not the variability.

## What `pc-fit` does with all of this

```bash
pc-fit --tic 410214986 --period 8.138268 --t0 1332.30991 --duration 3.4 \
       --variability auto --out out/
```

`--variability auto` runs the GP and the local-baseline fit, disqualifies
either one that is not a measurement, and keeps whichever survives — the GP
unless it falls a factor of two short of the local depth. The rotation period
comes from the extraction's per-sector ACF, the window cap follows from it,
flares are masked before the hyperparameters are fitted, and both fits, the
amplitudes, and the reason for the choice all go into `out/<target>/fit.json`.

Next: [`02_running_the_tests.ipynb`](02_running_the_tests.ipynb) — the other
nineteen tests, on a real star.